# Demo: `backtest()` with LLM-powered explanations

This notebook demonstrates the `ForecastingAssistant.backtest()` method
combined with the `ask()` method to get natural-language explanations
of backtesting results using a Google Gemini LLM.

The workflow:
1. Configure the assistant with a Google API key
2. Profile the data
3. Generate a forecasting plan
4. Create a cross-validation strategy
5. Run backtesting
6. Use `ask()` to get LLM-powered explanations of the results

**Requirements:** A valid Google API key with access to Gemini models.

In [1]:
import sys
sys.path.insert(0, "..")

In [5]:
import warnings

import numpy as np
import pandas as pd
from skforecast.model_selection import TimeSeriesFold

from skforecast_ai import ForecastingAssistant

warnings.filterwarnings("ignore")

## 1. Configure the Assistant with Google Gemini

Replace the placeholder below with your actual Google API key.

In [ ]:
GOOGLE_API_KEY = "your-google-api-key-here"  # Replace with your actual Google API key

In [4]:
assistant = ForecastingAssistant(
    llm="google:gemini-3-flash-preview",
    api_key=GOOGLE_API_KEY,
)

## 2. Synthetic Data

Create sample datasets for single-series and multi-series backtesting.

In [6]:
rng = np.random.default_rng(42)
n = 365
dates = pd.date_range("2022-01-01", periods=n, freq="D")

# Single series target with trend + seasonality
target = 100 + np.cumsum(rng.normal(0, 1, n)) + 5 * np.sin(2 * np.pi * np.arange(n) / 7)

# Exogenous variables
promo = rng.normal(50, 10, n)
temperature = 15 + 10 * np.sin(2 * np.pi * np.arange(n) / 365) + rng.normal(0, 2, n)

# Single series with exogenous variables
df = pd.DataFrame({
    "date": dates,
    "sales": target,
    "promo": promo,
    "temperature": temperature,
})

# Multi-series (long format)
n_multi = 200
dates_multi = pd.date_range("2022-01-01", periods=n_multi, freq="D")
df_multi = pd.DataFrame({
    "date": np.tile(dates_multi, 3),
    "series_id": ["store_A"] * n_multi + ["store_B"] * n_multi + ["store_C"] * n_multi,
    "revenue": np.concatenate([
        100 + np.cumsum(rng.normal(0, 1, n_multi)),
        200 + np.cumsum(rng.normal(0, 1.5, n_multi)),
        150 + np.cumsum(rng.normal(0, 0.8, n_multi)),
    ]),
    "promo": rng.normal(50, 10, n_multi * 3),
})

print(f"df:       {df.shape}")
print(f"df_multi: {df_multi.shape}")
display(df.head())

df:       (365, 4)
df_multi: (600, 4)


,date,sales,promo,temperature
0,2022-01-01,100.304717,64.814555,15.895627
1,2022-01-02,103.173890,42.564118,15.288682
2,2022-01-03,104.889824,41.777500,16.441694
3,2022-01-04,103.125168,52.023062,15.140855
4,2022-01-05,96.835295,58.443852,16.244312


---
## 3. Single Series Backtesting + LLM Explanation

Run backtesting on a single series with `ForecasterRecursive`, then use `ask()` to get an LLM-generated interpretation of the results.

### 3.1 Profile, Plan, and Backtest

In [7]:
profile = assistant.profile(
    data        = df,
    target      = "sales",
    date_column = "date",
)

plan = assistant.plan(
    profile    = profile,
    steps      = 14,
    forecaster = "ForecasterRecursive",
    estimator  = "LGBMRegressor",
)

cv, cv_explanation = assistant.create_cv(
    profile = profile,
    plan    = plan,
)

print("CV explanation:")
print(cv_explanation)
print(f"\nCV: {cv}")

CV explanation:
Initial training up to 2022-09-12, expanding window, refit every fold, 14-step horizon.

CV: ============== 
TimeSeriesFold 
Initial train size    = 2022-09-12,
Steps                 = 14,
Fold stride           = 14,
Overlapping folds     = False,
Window size           = None,
Differentiation       = None,
Refit                 = True,
Fixed train size      = False,
Gap                   = 0,
Skip folds            = None,
Allow incomplete fold = True,
Return all indexes    = False,
Verbose               = False



In [13]:
profile = assistant.profile(
    data        = df,
    target      = "sales",
    date_column = "date",
)

plan = assistant.plan(
    profile    = profile,
    steps      = 14,
    forecaster = "ForecasterRecursive",
    estimator  = "LGBMRegressor",
)

cv, cv_explanation = assistant.create_cv(
    profile = profile,
    plan    = plan,
    prompt  = "Initial training up to 2022-09-12, expanding window, refit every fold, 14-step horizon"
)

print("CV explanation:")
print(cv_explanation)
print(f"\nCV: {cv}")

CV explanation:
The user specified a specific calendar date for the initial training split ('2022-09-12'). An expanding window strategy (fixed_train_size=False) was selected as requested to incorporate all historical data as the model progresses. Refitting every fold (refit=True) ensures the model is updated with the most recent observations before each 14-day forecast, simulating a continuous retraining deployment scenario. Initial training up to 2022-09-12, expanding window, refit every fold, 14-step horizon.

CV: ============== 
TimeSeriesFold 
Initial train size    = 2022-09-12,
Steps                 = 14,
Fold stride           = 14,
Overlapping folds     = False,
Window size           = None,
Differentiation       = None,
Refit                 = True,
Fixed train size      = False,
Gap                   = 0,
Skip folds            = None,
Allow incomplete fold = True,
Return all indexes    = False,
Verbose               = False



In [14]:
profile = assistant.profile(
    data        = df,
    target      = "sales",
    date_column = "date",
)

plan = assistant.plan(
    profile    = profile,
    steps      = 14,
    forecaster = "ForecasterRecursive",
    estimator  = "LGBMRegressor",
)

cv, cv_explanation = assistant.create_cv(
    profile = profile,
    plan    = plan,
    prompt  = "I need to predict 14 steps every week. I get new data all sundays and want to use all my data for training."
)

print("CV explanation:")
print(cv_explanation)
print(f"\nCV: {cv}")

CV explanation:
Based on your requirement to predict every week (fold_stride=7) for a 14-step horizon, and your desire to use all available data (fixed_train_size=False), I've configured an expanding window strategy. The initial_train_size is set to 250 to exceed the minimum requirement of 236 (dictated by your max_lag of 118) while still allowing for 15+ folds of evaluation. Refit is set to True to simulate retraining the model each time new weekly data arrives. Using 68% of data (250 observations) for initial training, expanding window, refit every fold, 14-step horizon, 17 folds.

CV: ============== 
TimeSeriesFold 
Initial train size    = 250,
Steps                 = 14,
Fold stride           = 7,
Overlapping folds     = True,
Window size           = None,
Differentiation       = None,
Refit                 = True,
Fixed train size      = False,
Gap                   = 0,
Skip folds            = None,
Allow incomplete fold = True,
Return all indexes    = False,
Verbose             

In [8]:
result = assistant.backtest(
    data        = df,
    target      = "sales",
    date_column = "date",
    cv          = cv,
    profile     = profile,
    plan        = plan,
)

print("Explanation:")
print(result.explanation)
print("\nMetrics:")
display(result.metrics)
print("\nPredictions (first rows):")
display(result.predictions.head(10))

100%|██████████| 8/8 [00:00<00:00, 14.44it/s]

Explanation:
Initial training up to 2022-09-12, expanding window, refit every fold, 14-step horizon, 8 folds. Results — mean_absolute_error: 3.1849, mean_squared_error: 14.4905, mean_absolute_scaled_error: 1.0822, mean_absolute_percentage_error: 0.0340.

Metrics:


,mean_absolute_error,mean_squared_error,mean_absolute_scaled_error,mean_absolute_percentage_error
0,3.184862,14.490499,1.082238,0.034049



Predictions (first rows):


,fold,pred
2022-09-13,0,93.641670
2022-09-14,0,90.002343
2022-09-15,0,85.876926
2022-09-16,0,83.741837
2022-09-17,0,87.837475
2022-09-18,0,91.933170
2022-09-19,0,94.988806
2022-09-20,0,93.758693
2022-09-21,0,89.148648
2022-09-22,0,84.682383


In [9]:
print(result.code)

import pandas as pd
from lightgbm import LGBMRegressor
from skforecast.preprocessing import RollingFeatures
from skforecast.recursive import ForecasterRecursive
from skforecast.model_selection import TimeSeriesFold, backtesting_forecaster

# Load data
data = pd.read_csv('data.csv')

data['date'] = pd.to_datetime(data['date'])
data = data.set_index('date')
data = data.asfreq('D')
data = data.sort_index()

window_features = RollingFeatures(
    stats        = ['mean', 'mean'],
    window_sizes = [7, 91],
)

# Create forecaster
forecaster = ForecasterRecursive(
    estimator            = LGBMRegressor(random_state=123, verbose=-1),
    lags                 = [1, 3, 7, 8, 10, 12, 118],
    window_features      = window_features,
    categorical_features = 'auto',
    dropna_from_series   = False,
)

# Time series cross-validation configuration
cv = TimeSeriesFold(
    steps              = 14,
    initial_train_size = '2022-09-12',
    refit              = True,
    fixed_train_size   = Fal

### 3.2 Ask the LLM about the backtesting results

Pass the `backtest_result` to `ask()` — the LLM receives metrics, predictions, and CV configuration in context.

In [11]:
answer = assistant.ask(
    prompt="Analyze the backtesting results. Are the metrics good? "
           "What does the error distribution suggest about the model?",
    backtest_result=result,
)
print(answer.explanation)

### Analysis of Backtesting Results

The evaluation metrics provide a comprehensive view of how the `ForecasterRecursive` model with `LGBMRegressor` performed across the backtesting folds.

#### Evaluation Metrics Interpretation

*   **MASE (1.08):** This is the most significant metric here. A Mean Absolute Scaled Error greater than 1 indicates that the model is performing **slightly worse than a naive baseline** (which simply predicts the next value based on the previous observation). Despite the sophistication of the LightGBM model and exogenous features, the simple persistence of the series is a strong competitor that the current configuration hasn't quite surpassed.
*   **MAPE (3.4%):** In absolute percentage terms, the error is quite low. On average, predictions are off by only about 3.4%. This suggests that while the model isn't beating the naive baseline, it is still producing forecasts that are very close to the actual values in terms of magnitude.
*   **MAE (3.18) vs. MSE (14.

In [12]:
answer = assistant.ask(
    prompt="Based on these backtesting results, what would you recommend "
           "to improve the forecast accuracy? Should I try different lags, "
           "add more features, or switch to a different model?",
    backtest_result=result,
)
print(answer.explanation)

Based on the backtesting results, your model is performing reasonably well with a **MAPE of 3.4%**, indicating a high level of accuracy in percentage terms. However, the **MASE of 1.08** is a critical signal: a value greater than 1 means your current machine learning model is slightly less accurate than a naive baseline (simply predicting the value from the previous period).

Here are recommendations to improve accuracy based on your current configuration:

### 1. Refine Lags and Window Features
Your current lag selection includes a very distant lag (**118**). In a dataset of only 365 observations, such a high lag significantly reduces the available training data and may introduce noise.
*   **Recommendation:** Perform a formal autocorrelation analysis (ACF/PACF). For daily sales, focus on lags that capture weekly seasonality (7, 14, 21) rather than scattered individual days like 8, 10, or 12.
*   **Window Features:** You are using 7-day and 91-day means. Since sales often depend on vo

---
## 4. Multi-Series Backtesting + LLM Explanation

Run backtesting on multiple series with `ForecasterRecursiveMultiSeries`.

In [ ]:
profile_multi = assistant.profile(
    data             = df_multi,
    target           = "revenue",
    date_column      = "date",
    series_id_column = "series_id",
)

plan_multi = assistant.plan(
    profile    = profile_multi,
    steps      = 7,
    forecaster = "ForecasterRecursiveMultiSeries",
    estimator  = "LGBMRegressor",
)

cv_multi, _ = assistant.create_cv(
    profile = profile_multi,
    plan    = plan_multi,
)

result_multi = assistant.backtest(
    data             = df_multi,
    target           = "revenue",
    date_column      = "date",
    series_id_column = "series_id",
    cv               = cv_multi,
    profile          = profile_multi,
    plan             = plan_multi,
)

print("Metrics (per series):")
display(result_multi.metrics)
print("\nPredictions (first rows):")
display(result_multi.predictions.head(10))

In [ ]:
answer_multi = assistant.ask(
    prompt="Compare the backtesting performance across the three stores. "
           "Which store is hardest to forecast and why might that be?",
    backtest_result=result_multi,
)
print(answer_multi.explanation)

---
## 5. General Forecasting Q&A (no data)

The `ask()` method can also answer general skforecast questions without any data context.

In [ ]:
answer_general = assistant.ask(
    prompt="What is the difference between ForecasterRecursive and "
           "ForecasterDirect? When should I use each one?",
)
print(answer_general.explanation)

In [ ]:
answer_intervals = assistant.ask(
    prompt="How do I add prediction intervals to my backtesting? "
           "What methods are available in skforecast?",
)
print(answer_intervals.explanation)

---
## 6. Backtesting with Prediction Intervals + LLM Analysis

Run backtesting with prediction intervals and ask the LLM to evaluate coverage.

In [ ]:
plan_intervals = assistant.plan(
    profile    = profile,
    steps      = 14,
    forecaster = "ForecasterRecursive",
    estimator  = "LGBMRegressor",
    interval   = [10, 90],
)

cv_intervals, _ = assistant.create_cv(
    profile = profile,
    plan    = plan_intervals,
)

result_intervals = assistant.backtest(
    data        = df,
    target      = "sales",
    date_column = "date",
    cv          = cv_intervals,
    profile     = profile,
    plan        = plan_intervals,
)

print("Metrics:")
display(result_intervals.metrics)
print("\nPredictions with intervals (first rows):")
display(result_intervals.predictions.head(10))

In [ ]:
answer_coverage = assistant.ask(
    prompt="Evaluate the prediction intervals. Is the coverage close "
           "to the nominal 80%? Are the intervals too wide or too narrow?",
    backtest_result=result_intervals,
)
print(answer_coverage.explanation)

---
## 7. Shortcut: Fully Automatic Backtesting

The `backtest()` method can handle everything automatically — profiling, plan generation, and execution — with just the data and a `TimeSeriesFold`.

In [ ]:
cv_simple = TimeSeriesFold(
    steps=14,
    initial_train_size=300,
    refit=False,
    fixed_train_size=False,
)

result_auto = assistant.backtest(
    data        = df,
    target      = "sales",
    date_column = "date",
    cv          = cv_simple,
)

print(f"Auto-selected forecaster: {result_auto.plan.forecaster}")
print(f"Auto-selected estimator: {result_auto.plan.estimator}")
print(f"Uses exog: {result_auto.plan.use_exog}")
print("\nMetrics:")
display(result_auto.metrics)

In [ ]:
answer_auto = assistant.ask(
    prompt="Summarize the full backtesting pipeline: what forecaster was chosen, "
           "what features were used, and how did it perform?",
    backtest_result=result_auto,
)
print(answer_auto.explanation)